In [12]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml import command, load_component
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import Input, Output
from azureml.core import Workspace, Dataset, Datastore

subscription_id = '813ac1cb-7b8e-45ab-9b3c-8510079c761a'
resource_group = 'MIBLab_Official'
workspace_name = 'miblab-official-ml'

workspace = Workspace(subscription_id, resource_group, workspace_name)
datastore = Datastore.get(workspace, "eegfoundationmodeldata")

# Get a handle to the workspace
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)

cluster_name = "Standard-FX24mds"
preprocessing_component = load_component(source="./TUHProcessingJob.yml")

In [13]:
input_dir = Input(
	type="uri_folder",
	mode='ro_mount',
	path="azureml://subscriptions/813ac1cb-7b8e-45ab-9b3c-8510079c761a/resourcegroups/MIBLab_Official/workspaces/miblab-official-ml/datastores/eegfoundationmodeldata/paths/TUH_BIDS_Converted/",
)

output_dir = Output(
	type="uri_folder",
	path='azureml://subscriptions/813ac1cb-7b8e-45ab-9b3c-8510079c761a/resourcegroups/MIBLab_Official/workspaces/miblab-official-ml/datastores/eegfoundationmodeldata/paths/TUH-Processed/',
	mode='rw_mount'
)

In [14]:
# Define a pipeline with component
@pipeline(default_compute=cluster_name)
def TUHPreprocessing(input_dir):
	preprocessing = preprocessing_component(input_dir=input_dir)
	return {
        "output_dir": preprocessing.outputs.output_dir
    }

In [15]:
tuhpreprocessing_job = TUHPreprocessing(input_dir=input_dir)
tuhpreprocessing_job.outputs.output_dir = output_dir

In [16]:
# submit job to workspace
tuhpreprocessing_job = ml_client.jobs.create_or_update(
	tuhpreprocessing_job,
    experiment_name="EEGFoundationModel",
)

tuhpreprocessing_job

Uploading Preprocessing (0.1 MBs): 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 99610/99610 [00:00<00:00, 1681408.17it/s]


Experiment,Name,Type,Status,Details Page
EEGFoundationModel,sincere_guava_m8j6rn7ys9,pipeline,NotStarted,Link to Azure Machine Learning studio
